## DBTC-1286_dexposure

In [0]:
env = 'p'

spark.sql(f"""
WITH joined AS (
       SELECT distinct
            DebtorCDI,
            CASE 
                WHEN SecUnsec = 'Unsecured D' THEN TRUE
                WHEN SecUnsec = 'Secured D' THEN FALSE
            END AS IsSecuredFlag,
            BusinessDate
        FROM `{env}-cdh`.derivativeactive.dexposure dexp
        LEFT JOIN `{env}-reportinputfiles`.investor.servicer_report_securedperimeter sec_per
        ON sec_per.DebtorCDI = dexp.CDI
            AND sec_per.ReferenceDate <= dexp.BusinessDate
        WHERE dexp.PortfolioName ILIKE '%monza%'
)

MERGE INTO `{env}-cdh`.derivativeactive.dexposure dexp
USING joined 
ON dexp.CDI = joined.DebtorCDI
    AND dexp.BusinessDate = joined.BusinessDate
    AND dexp.SourceSystem = 'profits'
WHEN MATCHED THEN UPDATE SET dexp.IsSecuredFlag = joined.IsSecuredFlag
""")